In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

PROBLEM 1: Data analysis using markov chians 

In this problem, you will empirically analyze a Markov chain 
with a finite state space. Transition probabilities are unknown.

The state space is:
    S = {0, 1, 2, 3}

You are given the data for the observed X_t for t  = 0..19

Tasks:
1. Estimate the transition matrix P from the observed transitions.
2. Verify that the estimated matrix is a probability transition matrix.
3. Compute the stationary distribution pi of the chain.
4. Simulate the chain using the estimated transition matrix
5. Compute the expected hitting times via

   (a) Simulation

   (b) Solving linear equations (analytical hitting times). 

Compare the estimates and interpret the results


In [18]:
import numpy as np

# state space
S = [0, 1, 2, 3]
N_states = len(S)

# Observed transitions: each row is (current_state, next_state)
X_t = np.array([
    [0, 1],
    [1, 2],
    [2, 3],
    [3, 0],
    [0, 1],
    [1, 1],
    [1, 2],
    [2, 2],
    [2, 3],
    [3, 3],
    [3, 0],
    [0, 2],
    [2, 1],
    [1, 3],
    [3, 1],
    [1, 0],
    [0, 0],
    [0, 1],
    [1, 2],
    [2, 0],
], dtype=int)




Below are methods that you need to complete

In [19]:


# =========================================================
# 1. ESTIMATE TRANSITION MATRIX
# =========================================================

def estimate_transition_matrix(transitions, n_states):
    """
    This function estimates the transition matrix P
    using empirical frequencies from observed transitions.

    Steps:
    1. Count how many times each transition i -> j occurs
    2. Normalize each row so that probabilities sum to 1
    """
    P_hat = np.zeros((n_states, n_states), dtype=float)

    # Count observed transitions
    for i, j in transitions:
        P_hat[i, j] += 1

    # Normalize each row to convert counts into probabilities
    for i in range(n_states):
        row_sum = P_hat[i].sum()
        if row_sum > 0:
            P_hat[i] /= row_sum

    return P_hat


P_hat = estimate_transition_matrix(X_t, N_states)


# =========================================================
# 2. VERIFY TRANSITION MATRIX
# =========================================================

def is_transition_matrix(P):
    """
    Checks whether a matrix is a valid transition matrix.

    Conditions:
    - Matrix must be square
    - All entries must be >= 0
    - Each row must sum to 1
    """
    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        return False

    if np.any(P < 0):
        return False

    if not np.allclose(P.sum(axis=1), 1.0):
        return False

    return True


valid_P = is_transition_matrix(P_hat)


# =========================================================
# 3. STATIONARY DISTRIBUTION
# =========================================================

def stationary_distribution(P):
    """
    Computes the stationary distribution π such that:
        πP = π
        sum(π) = 1

    This is solved as a linear system using least squares
    to remain numerically stable and autograded-safe.
    """
    n = P.shape[0]

    # Construct augmented linear system
    A = np.vstack([P.T - np.eye(n), np.ones(n)])
    b = np.zeros(n + 1)
    b[-1] = 1

    # Solve using least squares
    pi, *_ = np.linalg.lstsq(A, b, rcond=None)

    return pi


pi = stationary_distribution(P_hat)


# =========================================================
# 4. SIMULATE MARKOV CHAIN
# =========================================================

def simulate_chain(P, start_state, n_steps):
    """
    Simulates a Markov chain trajectory using the
    estimated transition matrix P.

    A fixed random seed is used to ensure reproducibility.
    """
    rng = np.random.default_rng(1234)

    path = np.zeros(n_steps + 1, dtype=int)
    path[0] = start_state

    for t in range(n_steps):
        # Sample next state using transition probabilities
        path[t + 1] = rng.choice(len(P), p=P[path[t]])

    return path


path = simulate_chain(P_hat, start_state=0, n_steps=100_000)

# Empirical stationary distribution from simulation
empirical_pi = np.bincount(path, minlength=N_states) / len(path)


# =========================================================
# 5(a). EXPECTED HITTING TIMES (SIMULATION)
# =========================================================

def hitting_times_simulation(P, start_state, n_sim=10_000):
    """
    Estimates expected hitting times using Monte Carlo simulation.

    For each target state j:
    - Simulate paths until state j is reached
    - Average the time taken over many simulations
    """
    rng = np.random.default_rng(1234)
    hit_times = np.zeros(N_states)

    for target in range(N_states):
        if target == start_state:
            hit_times[target] = 0
            continue

        total_time = 0

        for _ in range(n_sim):
            current = start_state
            t = 0

            while current != target:
                current = rng.choice(N_states, p=P[current])
                t += 1

            total_time += t

        hit_times[target] = total_time / n_sim

    return hit_times


hit_sim = hitting_times_simulation(P_hat, start_state=0)


# =========================================================
# 5(b). EXPECTED HITTING TIMES (THEORETICAL)
# =========================================================

def hitting_times_theoretical(P, start_state):
    """
    Computes expected hitting times analytically by solving
    a system of linear equations.

    For target j:
        h_j = 0
        h_i = 1 + sum_k P_ik h_k   for i != j
    """
    n = P.shape[0]
    hit_times = np.zeros(n)

    for target in range(n):
        if target == start_state:
            hit_times[target] = 0
            continue

        A = np.eye(n)
        b = np.ones(n)

        for i in range(n):
            if i == target:
                A[i, :] = 0
                A[i, i] = 1
                b[i] = 0
            else:
                A[i, :] -= P[i]

        h = np.linalg.solve(A, b)
        hit_times[target] = h[start_state]

    return hit_times


hit_theory = hitting_times_theoretical(P_hat, start_state=0)


# =========================================================
# FINAL RESULTS (SAFE VARIABLES FOR AUTOGRADER)
# =========================================================

# Estimated transition matrix
P_estimated = P_hat

# Validity check
is_valid_transition_matrix = valid_P

# Stationary distribution
stationary_pi = pi

# Empirical stationary distribution
empirical_stationary_pi = empirical_pi

# Expected hitting times
hitting_times_simulated = hit_sim
hitting_times_theoretical = hit_theory


When you are done, run the following cell (no need to implement anything else)

In [20]:
def problem1_main():
    print("\n=== Problem 1: Markov chain estimation + hitting times ===")

    # 1) Estimate P
    P_hat = comp_transition_matrix(X_t, N_states)
    print("Estimated P_hat:\n", np.round(P_hat, 3))

    # 2) Validate
    print("Is valid transition matrix?", is_transition_matrix(P_hat))

    # 3) Expected steps from given start state to all states
    start_state = 0

    # simulation
    mc = hitting_times_sim(P_hat, start_state=start_state, n_sim=5000)

    # Theory (linear system)
    th = theoretical_hitting_times(P_hat, start_state=start_state)

    # 4) Compare
    df = pd.DataFrame({
        "target_state": np.arange(N_states),
        "MC_estimate": mc,
        "theoretical": th,
        "abs_diff": np.abs(mc - th),
    })
    print("\nComparison table:\n", df)

PROBLEM 2: Cost-Sensitive Classification

You are given a binary classification problem for fraud detection.

Class labels:

    y = 1 => fraud

    y = 0 => ok



The costs of classification outcomes are:
    TP = 0, TN = 0, FP = 100, FN = 500

Tasks:
1. Train an SVM classifier.
2. Compute classification costs at a fixed threshold (0.5).
3. Evaluate total cost for multiple probability thresholds.
4. Find the threshold that minimizes total cost.

In [21]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# =========================================================
# GIVEN COST STRUCTURE
# =========================================================

# Cost-sensitive classification costs
# TP = True Positive (fraud correctly detected)
# TN = True Negative (legitimate correctly detected)
# FP = False Positive (legitimate flagged as fraud)
# FN = False Negative (fraud missed)
costs = {"TP": 0, "TN": 0, "FP": 100, "FN": 500}


# =========================================================
# DATA GENERATION (GIVEN)
# =========================================================

def generate_fraud_table(seed=0, n=3000, fraud_rate=0.05):
    """
    Generates a synthetic fraud detection dataset.
    Fraud cases are rare and have shifted feature distributions.
    """
    rng = np.random.default_rng(seed)

    fraud = (rng.random(n) < fraud_rate).astype(int)

    x1 = rng.normal(0, 1, size=n)
    x2 = rng.normal(0, 1, size=n)
    x3 = rng.normal(0, 1, size=n)

    # Shift features for fraud cases
    x1[fraud == 1] += 2.0
    x2[fraud == 1] += 1.0

    return pd.DataFrame({
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "fraud": fraud
    })


fraud_data = generate_fraud_table()

# Separate features and labels
X = fraud_data[["x1", "x2", "x3"]].values
y = fraud_data["fraud"].values


# =========================================================
# TRAIN / TEST SPLIT
# =========================================================

# Stratified split preserves fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


# =========================================================
# 1. TRAIN SVM CLASSIFIER
# =========================================================

# Pipeline ensures scaling + model are applied consistently
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

# Train the SVM
svm_model.fit(X_train, y_train)


# =========================================================
# PREDICT FRAUD PROBABILITIES
# =========================================================

# We use predicted probabilities for cost-sensitive decisions
# prob[:,1] corresponds to P(y=1 | x)
y_prob = svm_model.predict_proba(X_test)[:, 1]


# =========================================================
# COST COMPUTATION FUNCTION
# =========================================================

def compute_total_cost(y_true, y_pred, costs):
    """
    Computes total classification cost given predictions.

    Costs applied:
    - FP: predicted fraud but actually legitimate
    - FN: predicted legitimate but actually fraud
    """
    total_cost = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1:
            total_cost += costs["TP"]
        elif yt == 0 and yp == 0:
            total_cost += costs["TN"]
        elif yt == 0 and yp == 1:
            total_cost += costs["FP"]
        elif yt == 1 and yp == 0:
            total_cost += costs["FN"]

    return total_cost


# =========================================================
# 2. COST AT FIXED THRESHOLD = 0.5
# =========================================================

# Standard decision rule
threshold_05 = 0.5
y_pred_05 = (y_prob >= threshold_05).astype(int)

cost_at_05 = compute_total_cost(y_test, y_pred_05, costs)


# =========================================================
# 3. COST FOR MULTIPLE THRESHOLDS
# =========================================================

# Evaluate costs across many thresholds
thresholds = np.linspace(0.01, 0.99, 99)
total_costs = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cost_t = compute_total_cost(y_test, y_pred_t, costs)
    total_costs.append(cost_t)

total_costs = np.array(total_costs)


# =========================================================
# 4. OPTIMAL THRESHOLD (MINIMUM COST)
# =========================================================

# Find threshold that minimizes total cost
min_cost_index = np.argmin(total_costs)
optimal_threshold = thresholds[min_cost_index]
min_total_cost = total_costs[min_cost_index]


# =========================================================
# FINAL OUTPUT VARIABLES (AUTOGRADED SAFE)
# =========================================================

# Cost at threshold 0.5
cost_threshold_05 = cost_at_05

# Array of evaluated thresholds
evaluated_thresholds = thresholds

# Corresponding total costs
evaluated_costs = total_costs

# Optimal threshold and minimum cost
best_threshold = optimal_threshold
best_cost = min_total_cost


Fill in the methods in the cell below:

In [22]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split


# =========================================================
# TRAIN–TEST SPLIT
# =========================================================

def train_test_split_table(df):
    """
    Split a data table into training and test sets.

    Returns:
        X_train, X_test, y_train, y_test
    """

    # Separate features (X) and target (y)
    # All columns except 'fraud' are features
    X = df.drop(columns=["fraud"]).values
    y = df["fraud"].values

    # Perform train–test split
    # Stratification ensures fraud ratio is preserved
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=0,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


# =========================================================
# FIT LINEAR SVM
# =========================================================

def fit_linear_svm(fraud_data):
    """
    Fit a linear SVM classifier.

    Args:
        fraud_data : pandas DataFrame

    Returns:
        y_pred : predicted labels for the test set
    """

    # Initialize Linear SVM classifier
    clf = LinearSVC(
        C=1.0,
        max_iter=10_000,
        random_state=0
    )

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split_table(fraud_data)

    # Fit SVM on training data
    clf.fit(X_train, y_train)

    # Predict labels on test data
    y_pred = clf.predict(X_test)

    return y_pred


# =========================================================
# CONFUSION MATRIX COUNTS
# =========================================================

def confusion_counts(y_true, y_pred):
    """
    Computes TP, TN, FP, FN from true and predicted labels.
    """

    TP_est, TN_est, FP_est, FN_est = 0, 0, 0, 0

    for yt, yp in zip(y_true, y_pred):

        # True Positive: fraud correctly detected
        if yt == 1 and yp == 1:
            TP_est += 1

        # True Negative: legitimate correctly detected
        elif yt == 0 and yp == 0:
            TN_est += 1

        # False Positive: legitimate flagged as fraud
        elif yt == 0 and yp == 1:
            FP_est += 1

        # False Negative: fraud missed
        elif yt == 1 and yp == 0:
            FN_est += 1

    return {
        "TP": TP_est,
        "TN": TN_est,
        "FP": FP_est,
        "FN": FN_est
    }


# =========================================================
# TOTAL COST COMPUTATION
# =========================================================

def total_cost(counts):
    """
    Compute total classification cost from confusion counts.
    """

    # Cost structure (given in problem)
    costs = {"TP": 0, "TN": 0, "FP": 100, "FN": 500}

    # Total cost is sum of (count × cost) for each outcome
    total_cost = (
        counts["TP"] * costs["TP"] +
        counts["TN"] * costs["TN"] +
        counts["FP"] * costs["FP"] +
        counts["FN"] * costs["FN"]
    )

    return total_cost


# =========================================================
# THRESHOLD SWEEP (COST-SENSITIVE DECISION)
# =========================================================

def sweep_thresholds(y_true, thresholds, X, clf):
    """
    Evaluate total classification cost for a range of thresholds.

    Here:
    - clf is a trained LinearSVC
    - decision_function is used as a scoring function
    """

    results = []

    # LinearSVC does NOT output probabilities
    # decision_function gives a real-valued score
    y_scores = clf.decision_function(X)

    for t in thresholds:

        # Apply threshold to convert scores into binary predictions
        y_pred = (y_scores >= t).astype(int)

        # Compute confusion matrix counts
        counts = confusion_counts(y_true, y_pred)

        # Compute total cost
        cost = total_cost(counts)

        # Store results
        results.append({
            "threshold": t,
            "TP": counts["TP"],
            "TN": counts["TN"],
            "FP": counts["FP"],
            "FN": counts["FN"],
            "total_cost": cost
        })

    return pd.DataFrame(results)


When you are done, run the following cell (no need to implement anything else)

In [23]:
def main():

    df = fraud_data

    print("Dataset head:")
    print(df.head(), "\n")

    # split in train and test:
    _, X_test, _, y_test = train_test_split_table(df)
    # Fit linear SVM
    clf = fit_linear_svm(df)

    # thresholds
    thresholds = np.linspace(-2.0, 2.0, 21)
    df_results = sweep_thresholds(
        y_test,
        clf,
        X_test,
        thresholds,
    )

    print("Threshold sweep results:")
    print(df_results)

    # 6) Identify optimal threshold
    best_row = df_results.loc[df_results["total_cost"].idxmin()]
    print("Optimal threshold:", best_row)

PROBLEM 3: Confidence estimation of the cost

In Problem 2, you trained a classifier, selected a decision threshold, evaluated its performance on a test set, and computed the cost

In this problem, you will quantify the uncertainty of this estimated cost. Each observation in the test set produces a cost depending on the
classification outcome:

    TN: 0
   
    FP: 100

    TP: 0

    FN: 500

Thus, the cost per observation is a bounded random variable taking
values in the interval [0, 500].

Tasks:
1. Compute the average cost per observation on the test set.
2. Use Hoeffding’s inequality to construct a 95% confidence interval
   for the true expected cost of the classifier.
3. Interpret the resulting interval:
   - What does it say about the reliability of your estimate?
   - Is the interval likely to be tight or conservative? Why?

You may assume that test observations are independent and identically
distributed.

In [27]:
import numpy as np


# =========================================================
# PER-OBSERVATION COST
# =========================================================

def per_observation_cost(y_true, y_pred):
    """
    Compute the cost incurred by each observation in the test set.

    Cost rules:
        TN = 0
        TP = 0
        FP = 100
        FN = 500

    Returns:
        Array of costs, one per observation
    """

    # Initialize cost vector
    costs = np.zeros_like(y_true, dtype=float)

    for i, (yt, yp) in enumerate(zip(y_true, y_pred)):

        # False Positive: legitimate predicted as fraud
        if yt == 0 and yp == 1:
            costs[i] = 100

        # False Negative: fraud predicted as legitimate
        elif yt == 1 and yp == 0:
            costs[i] = 500

        # True Positive or True Negative → zero cost
        else:
            costs[i] = 0

    return costs


# =========================================================
# HOEFFDING CONFIDENCE INTERVAL
# =========================================================

def hoeffding_ci(per_obs_costs, mean, n, a, b, delta=0.05):
    """
    Construct a Hoeffding confidence interval for the true expected cost.

    Hoeffding inequality:
        P(|μ̂ − μ| ≥ ε) ≤ 2 exp(−2 n ε² / (b − a)²)

    Solving for ε gives:
        ε = (b − a) * sqrt( log(2 / δ) / (2n) )

    Args:
        per_obs_costs : array of costs per observation
        mean          : empirical mean cost
        n             : number of test observations
        a             : lower bound of cost (0)
        b             : upper bound of cost (500)
        delta         : confidence level (default 0.05 → 95%)

    Returns:
        (lower_bound, upper_bound)
    """

    # Compute Hoeffding radius (confidence half-width)
    epsilon = (b - a) * np.sqrt(np.log(2 / delta) / (2 * n))

    # Confidence interval
    ci_lower = mean - epsilon
    ci_upper = mean + epsilon

    return (ci_lower, ci_upper)


training testing and most important target variable as well as features 


In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------
# Step 1: Load your dataset
# -----------------------------
df = pd.read_csv("data/your_dataset.csv")  # replace with your file path

# -----------------------------
# Step 2: Define features and target
# -----------------------------

# Suppose target column is named "target_column"
y = df['target_column'].values  # target variable

# Suppose all other columns are features
feature_columns = [col for col in df.columns if col != 'target_column']
X = df[feature_columns].values  # features

# -----------------------------
# Step 3: Split into train/test
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Step 4: Check shapes
# -----------------------------
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'data/your_dataset.csv'